In [ ]:
# Simulate on real quantum computer
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager # Thêm thư viện dịch mạch
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

service = QiskitRuntimeService(name="trandon")

# 1. Tạo mạch lượng tử gốc (Ý tưởng của bạn)
qc = QuantumCircuit(2, 2)
qc.h(0)          
qc.cx(0, 1)      
qc.measure([0, 1], [0, 1]) 

# 2. Tìm máy lượng tử thật đang rảnh nhất
print("\nSearching for the most optimal real quantum device...")
backend = service.least_busy(simulator=False, operational=True)
print("=" * 50)
print(f"Running on real quantum hardware: {backend.name}")
print("=" * 50)
# === BƯỚC SỬA LỖI: TRANSPILE (DỊCH MẠCH) ===
print("\nTranslating the circuit for the real quantum device...")
pm = generate_preset_pass_manager(optimization_level=1, backend=backend)
isa_circuit = pm.run(qc) # Đây là mạch đã được tối ưu cho máy thật
print("Translating success")
# ============================================

# 3. Gửi mạch ĐÃ DỊCH (isa_circuit) lên máy thật
print("\nSending to IBM Quantum...")

sampler = SamplerV2(mode=backend) 
job = sampler.run([isa_circuit], shots=1024) # Gửi isa_circuit thay vì qc

print(f"Job ID: {job.job_id()}")

result = job.result()
pub_result = result[0]

# 4. In kết quả
counts = pub_result.data.c.get_counts()
print("\n===== RESULTS FROM REAL QUANTUM HARDWARE =====")
print(counts)

KeyboardInterrupt: 

In [5]:
# Simulate on real quantum computer
import os
import pickle
import matplotlib.pyplot as plt
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit.library import U3Gate
from qiskit.quantum_info import DensityMatrix, Statevector, partial_trace
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2

# 1. Khởi tạo Service và chọn Backend thực tế
service = QiskitRuntimeService(name="trandon")

# Tự động chọn backend thực tế ít bận nhất (least busy) có tối thiểu 5 qubits
backend = service.least_busy(
    operational=True, simulator=False, min_num_qubits=5
)
print(f"Đã chọn Backend thực tế: {backend.name}")

# 2. Định nghĩa hàm tạo mạch Zurek (1 Sys + 4 Env)
numQubits = 5
theta, phi = np.pi, 0


def create_zurek_circuit(theta, phi, n):
    qc = QuantumCircuit(n)
    qb_sys = 0
    qb_envir = list(range(1, n))

    qc.h(qb_sys)
    for i in qb_envir:
        cu_gate = U3Gate(theta, phi, 0).control(1)
        qc.append(cu_gate, [qb_sys, i])
    return qc


base_qc = create_zurek_circuit(theta, phi, numQubits)

# 3. Transpile mạch sang Native Gates của Backend thực tế
pm = generate_preset_pass_manager(target=backend.target, optimization_level=3)
transpiled_qc = pm.run(base_qc)

# 4. Thực thi mạch trên phần cứng IBM bằng SamplerV2
# Thêm phép đo toàn bộ qubits để quan sát phân bố xác suất
meas_qc = transpiled_qc.copy()
meas_qc.measure_all()

sampler = SamplerV2(mode=backend)
print("Đang gửi Job lên máy tính lượng tử thực tế...")
job = sampler.run([meas_qc], shots=8192)
print(f"Job ID: {job.job_id()}")

# Đợi lấy kết quả
result = job.result()
pub_result = result[0]
counts = pub_result.data.meas.get_counts()

print("Kết quả đo Counts từ máy thật:", counts)


# 5. Hàm tính VNE và QMI
def VNE(state):
    eigenvalues, _ = np.linalg.eigh(state)
    eigenvalues = eigenvalues[eigenvalues > 1e-12]
    return -np.sum(eigenvalues * np.log2(eigenvalues))


def quantum_mutual_I(S_State, E_State, SE_State):
    return VNE(S_State) + VNE(E_State) - VNE(SE_State)


# 6. Tính QMI lý thuyết/ideal (dùng làm baseline so sánh với thực nghiệm)
full_sv = Statevector.from_instruction(base_qc)
rho_S = partial_trace(full_sv, list(range(1, numQubits))).data

env_qubits = list(range(1, numQubits))
results_S_Envir = []

for k in range(1, len(env_qubits) + 1):
    Ef = env_qubits[:k]
    trace_out_Ef = [i for i in range(numQubits) if i not in Ef]
    trace_out_SEf = [i for i in range(numQubits) if i not in ([0] + Ef)]

    rho_Ef = partial_trace(full_sv, trace_out_Ef).data
    rho_SEf = partial_trace(full_sv, trace_out_SEf).data

    QMI_S_Ef = quantum_mutual_I(rho_S, rho_Ef, rho_SEf)
    results_S_Envir.append(QMI_S_Ef)

# 7. Lưu kết quả
data = {
    "counts_real_device": counts,
    "backend_name": backend.name,
    "results_S_Envir_ideal": results_S_Envir,
    "transpiled_circuit": transpiled_qc,
}

os.makedirs("Figure", exist_ok=True)
with open("QMI_RealHardware_Zurek.pkl", "wb") as f:
    pickle.dump(data, f)

print("=" * 50)
print(f"Hoàn tất chạy trên phần cứng: {backend.name}")
print("Đã lưu kết quả thực nghiệm vào QMI_RealHardware_Zurek.pkl")
print("=" * 50)

IBMInputValueError: 'No matching instances found for the following filters: .'